# Hyperparameter Tuning — Baseline + Deep Models
**Baseline:** GridSearchCV over LogReg / SVM / RF / GBM  
**Deep:** Optuna TPE search over FusionModel  
**Targets:** Valence · Arousal · Dominance  
Run on Google Colab (GPU recommended for deep tuning).

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_REPO = "https://github.com/YOUR_USERNAME/emotion-recognition.git"
    !git clone {GITHUB_REPO} /content/emotion-recognition
    %cd /content/emotion-recognition
    !pip install mat73 pyyaml scikit-learn optuna -q

    from google.colab import drive
    drive.mount("/content/drive")

    import shutil, os
    os.makedirs("data/raw", exist_ok=True)
    shutil.copy("/content/drive/MyDrive/DREAMER.mat", "data/raw/DREAMER.mat")
    print("✅ Setup complete")
else:
    print("✅ Running locally")

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, ".")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config

config      = load_config("configs/default.yaml")
TARGETS     = config["labels"]["targets"]          # valence, arousal, dominance
RESULTS_DIR = "outputs/results"
MODELS_DIR  = "outputs/models"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

sns.set_theme(style="darkgrid")
plt.rcParams.update({"figure.dpi": 120})
print("✅ Config loaded:", config["model"]["type"])

## PART A — Baseline Grid Search
Extract the full feature matrix once, then run grid search for each target.

In [ ]:
from tqdm.auto import tqdm
from src.data.loader import (
    load_dreamer_mat, get_subject_data,
    get_trial_signals, get_trial_labels,
    N_SUBJECTS, N_VIDEOS
)
from src.data.preprocessor     import process_trial
from src.features.eeg_features import extract_eeg_features
from src.features.ecg_features import extract_ecg_features

CACHE_X = os.path.join(RESULTS_DIR, "feature_matrix_X.npy")
CACHE_Y = os.path.join(RESULTS_DIR, "feature_matrix_y.npz")
CACHE_S = os.path.join(RESULTS_DIR, "feature_matrix_subjects.npy")

if os.path.exists(CACHE_X):
    print("✅ Loading cached feature matrix...")
    X        = np.load(CACHE_X)
    y_cache  = np.load(CACHE_Y)
    y        = {k: y_cache[k] for k in y_cache.files}
    subjects = np.load(CACHE_S)
    print(f"   X shape: {X.shape} | subjects: {np.unique(subjects)}")

else:
    print("⏳ Building feature matrix from scratch (~8–12 min)...")
    dreamer  = load_dreamer_mat(config["data"]["raw_path"])
    EEG_FS   = config["data"]["sampling_rate_eeg"]
    ECG_FS   = config["data"]["sampling_rate_ecg"]
    THR      = config["labels"]["threshold"]

    rows, sub_ids = [], []
    ys = {"valence": [], "arousal": [], "dominance": []}

    for sub_idx in tqdm(range(N_SUBJECTS), desc="Subjects"):
        subject = get_subject_data(dreamer, sub_idx)
        for vid_idx in range(N_VIDEOS):
            try:
                eeg_s, ecg_s = get_trial_signals(subject, vid_idx, "stimuli")
                eeg_b, ecg_b = get_trial_signals(subject, vid_idx, "baseline")
                raw_labels   = get_trial_labels(subject, vid_idx)
                bin_labels   = {k: int(v > THR) for k, v in raw_labels.items()}

                eeg_segs, ecg_segs = process_trial(
                    eeg_s, ecg_s, eeg_b, ecg_b,
                    eeg_fs=EEG_FS, ecg_fs=ECG_FS,
                    window_sec  = config["data"]["segment_length"],
                    overlap_sec = config["data"]["overlap"],
                    norm_method = config["data"]["norm_method"],
                )
                n = min(eeg_segs.shape[0], ecg_segs.shape[0])
                for w in range(n):
                    feat = np.concatenate([
                        extract_eeg_features(eeg_segs[w], fs=EEG_FS),
                        extract_ecg_features(ecg_segs[w], fs=ECG_FS),
                    ])
                    rows.append(feat)
                    sub_ids.append(sub_idx + 1)
                    for t in TARGETS:
                        ys[t].append(bin_labels[t])

            except Exception as e:
                print(f"  ⚠ sub={sub_idx+1} vid={vid_idx+1}: {e}")

    X        = np.stack(rows).astype(np.float32)
    y        = {t: np.array(ys[t]) for t in TARGETS}
    subjects = np.array(sub_ids)

    np.save(CACHE_X, X)
    np.savez(CACHE_Y, **y)
    np.save(CACHE_S, subjects)
    print(f"✅ Feature matrix saved | X: {X.shape}")

## A1 — Grid Search: Random Forest (recommended starting point)

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.ensemble        import RandomForestClassifier

RF_GRID = {
    "clf__n_estimators"     : [100, 200, 300],
    "clf__max_depth"        : [None, 10, 20, 30],
    "clf__min_samples_split": [2, 5, 10],
    "clf__max_features"     : ["sqrt", "log2"],
    "clf__class_weight"     : ["balanced"],
}

rf_grid_results = {}

for target in TARGETS:
    print(f"\n{'='*52}")
    print(f"  GridSearch RF | target = {target.upper()}")
    print(f"{'='*52}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    RandomForestClassifier(random_state=42, n_jobs=-1)),
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gs = GridSearchCV(
        pipe, RF_GRID, cv=cv,
        scoring="roc_auc", n_jobs=-1,
        verbose=1, refit=True,
    )
    gs.fit(X, y[target])

    result = {
        "target"      : target,
        "best_params" : gs.best_params_,
        "best_auc"    : round(gs.best_score_, 4),
        "cv_results"  : {
            "mean_test": gs.cv_results_["mean_test_score"].tolist(),
            "std_test" : gs.cv_results_["std_test_score"].tolist(),
        }
    }
    rf_grid_results[target] = result

    path = f"{RESULTS_DIR}/gridsearch_rf_{target}.json"
    with open(path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"  Best AUC   : {result['best_auc']}")
    print(f"  Best params: {result['best_params']}")

## A2 — Grid Search: SVM

In [ ]:
from sklearn.svm import SVC

SVM_GRID = {
    "clf__C"      : [0.1, 1.0, 10.0, 100.0],
    "clf__gamma"  : ["scale", "auto", 0.001, 0.01],
    "clf__kernel" : ["rbf", "linear"],
}

svm_grid_results = {}

for target in TARGETS:
    print(f"\n{'='*52}")
    print(f"  GridSearch SVM | target = {target.upper()}")
    print(f"{'='*52}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    SVC(probability=True, class_weight="balanced",
                       random_state=42)),
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gs = GridSearchCV(
        pipe, SVM_GRID, cv=cv,
        scoring="roc_auc", n_jobs=-1,
        verbose=1, refit=True,
    )
    gs.fit(X, y[target])

    result = {
        "target"      : target,
        "best_params" : gs.best_params_,
        "best_auc"    : round(gs.best_score_, 4),
    }
    svm_grid_results[target] = result

    path = f"{RESULTS_DIR}/gridsearch_svm_{target}.json"
    with open(path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"  Best AUC: {result['best_auc']} | {result['best_params']}")

## A3 — Grid Search: Logistic Regression + GBM

In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import GradientBoostingClassifier

LOGREG_GRID = {
    "clf__C"      : [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    "clf__penalty": ["l2"],
    "clf__solver" : ["lbfgs"],
}

GBM_GRID = {
    "clf__n_estimators"  : [100, 200, 300],
    "clf__learning_rate" : [0.01, 0.05, 0.1, 0.2],
    "clf__max_depth"     : [3, 4, 6],
    "clf__subsample"     : [0.7, 0.85, 1.0],
}

logreg_results, gbm_results = {}, {}

for target in TARGETS:
    for model_type, model_cls, grid, store in [
        ("logreg", LogisticRegression(max_iter=1000,
                                       class_weight="balanced",
                                       random_state=42),
         LOGREG_GRID, logreg_results),
        ("gbm",    GradientBoostingClassifier(random_state=42),
         GBM_GRID, gbm_results),
    ]:
        print(f"\n── {model_type.upper()} | {target.upper()} ──")
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    model_cls),
        ])
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        gs = GridSearchCV(
            pipe, grid, cv=cv,
            scoring="roc_auc", n_jobs=-1, verbose=0
        )
        gs.fit(X, y[target])

        result = {
            "target"      : target,
            "best_params" : gs.best_params_,
            "best_auc"    : round(gs.best_score_, 4),
        }
        store[target] = result
        path = f"{RESULTS_DIR}/gridsearch_{model_type}_{target}.json"
        with open(path, "w") as f:
            json.dump(result, f, indent=2)

        print(f"  Best AUC: {result['best_auc']} | {result['best_params']}")

## A4 — Baseline Grid Search Summary

In [ ]:
summary_rows = []
for target in TARGETS:
    for model_type, store in [
        ("rf",     rf_grid_results),
        ("svm",    svm_grid_results),
        ("logreg", logreg_results),
        ("gbm",    gbm_results),
    ]:
        if target in store:
            summary_rows.append({
                "target"    : target,
                "model"     : model_type.upper(),
                "best_auc"  : store[target]["best_auc"],
                "best_params": str(store[target]["best_params"]),
            })

df_gs = pd.DataFrame(summary_rows)
print("── Grid Search Summary ──")
print(df_gs.to_string(index=False))
df_gs.to_csv(f"{RESULTS_DIR}/gridsearch_summary.csv", index=False)

# ── Heatmap: best AUC per model × target ──────────────────────────
pivot = df_gs.pivot(index="model", columns="target", values="best_auc")

plt.figure(figsize=(8, 5))
sns.heatmap(
    pivot, annot=True, fmt=".3f",
    cmap="YlGn", vmin=0.45, vmax=1.0,
    linewidths=0.5, annot_kws={"size": 12}
)
plt.title("Grid Search — Best CV AUC per Model × Target",
          fontweight="bold")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/gridsearch_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
from src.models.baseline import save_model

ALL_GS = {
    "rf": rf_grid_results, "svm": svm_grid_results,
    "logreg": logreg_results, "gbm": gbm_results,
}

for target in TARGETS:
    # Find best model for this target
    best_model_type = max(
        ["rf", "svm", "logreg", "gbm"],
        key=lambda m: ALL_GS[m].get(target, {}).get("best_auc", 0)
    )
    best_params = ALL_GS[best_model_type][target]["best_params"]

    print(f"\n{target.upper()} → best: {best_model_type.upper()} "
          f"AUC={ALL_GS[best_model_type][target]['best_auc']}")

    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression

    clf_map = {
        "rf"    : RandomForestClassifier,
        "svm"   : SVC,
        "logreg": LogisticRegression,
        "gbm"   : GradientBoostingClassifier,
    }
    # Strip pipeline prefix from param keys
    clean_params = {
        k.replace("clf__", ""): v
        for k, v in best_params.items()
    }
    clf = clf_map[best_model_type](
        **clean_params, probability=True,
        random_state=42
    ) if best_model_type == "svm" else \
        clf_map[best_model_type](**clean_params, random_state=42)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    clf),
    ])
    pipe.fit(X, y[target])
    save_model(
        pipe,
        f"{MODELS_DIR}/best_tuned_{best_model_type}_{target}.pkl"
    )
    print(f"  Saved tuned model → {MODELS_DIR}/")

---
## PART B — Optuna Deep Model Tuning
Bayesian hyperparameter search for FusionModel.  
**GPU strongly recommended.** Each trial ≈ 2–4 min on CPU, ~30s on GPU.

In [ ]:
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from sklearn.metrics  import roc_auc_score

from src.data.dataset      import DREAMERDataset
from src.models.deep_model import FusionModel
from src.training.trainer  import make_weighted_sampler, _run_epoch

optuna.logging.set_verbosity(optuna.logging.WARNING)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Optuna search space ───────────────────────────────────────────
SEARCH_SPACE = {
    "lr"          : ("log_float", 1e-4, 1e-2),
    "batch_size"  : ("categorical", [16, 32, 64]),
    "branch_dim"  : ("categorical", [64, 128, 256]),
    "dropout"     : ("float", 0.2, 0.6),
    "weight_decay": ("log_float", 1e-5, 1e-3),
    "branch_type" : ("categorical", ["cnn", "cnnlstm"]),
    "window_sec"  : ("categorical", [2, 4]),
}
print("Search space defined:", list(SEARCH_SPACE.keys()))

In [ ]:
def make_objective(target: str):
    """Factory: returns an Optuna objective for a given target."""

    def objective(trial: optuna.Trial) -> float:
        # ── Sample hyperparameters ─────────────────────────────
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [16, 32, 64])
        branch_dim   = trial.suggest_categorical("branch_dim", [64, 128, 256])
        dropout      = trial.suggest_float("dropout", 0.2, 0.6)
        weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
        branch_type  = trial.suggest_categorical("branch_type",
                                                   ["cnn", "cnnlstm"])
        window_sec   = trial.suggest_categorical("window_sec", [2, 4])
        n_epochs     = trial.suggest_categorical("n_epochs", [15, 20, 25])

        # ── Build dataset ──────────────────────────────────────
        try:
            dataset = DREAMERDataset(
                mat_path    = config["data"]["raw_path"],
                target      = target,
                window_sec  = float(window_sec),
                overlap_sec = float(window_sec) / 2,
                norm_method = config["data"]["norm_method"],
                threshold   = float(config["labels"]["threshold"]),
            )
        except Exception as e:
            return 0.0

        n_val   = max(1, int(len(dataset) * 0.2))
        n_train = len(dataset) - n_val
        train_ds, val_ds = random_split(
            dataset, [n_train, n_val],
            generator=torch.Generator().manual_seed(42)
        )

        sampler      = make_weighted_sampler(dataset)
        train_loader = DataLoader(
            train_ds, batch_size=batch_size,
            sampler=sampler, num_workers=2,
            pin_memory=device.type == "cuda",
        )
        val_loader = DataLoader(
            val_ds, batch_size=batch_size,
            shuffle=False, num_workers=2,
        )

        # ── Build model ────────────────────────────────────────
        model = FusionModel(
            n_eeg_channels = 14,
            n_ecg_channels = 2,
            branch_type    = branch_type,
            branch_dim     = branch_dim,
            n_classes      = 2,
            dropout        = dropout,
        ).to(device)

        cw        = dataset.class_weights().to(device)
        criterion = nn.CrossEntropyLoss(weight=cw)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr, weight_decay=weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=n_epochs
        )

        # ── Training loop ──────────────────────────────────────
        best_auc      = 0.0
        patience_ctr  = 0
        PATIENCE      = 5

        for epoch in range(n_epochs):
            _run_epoch(model, train_loader, criterion,
                       optimizer, device, is_train=True)
            scheduler.step()

            # Collect val probs
            model.eval()
            y_true_v, y_prob_v = [], []
            with torch.no_grad():
                for eeg, ecg, labels in val_loader:
                    eeg, ecg   = eeg.to(device), ecg.to(device)
                    logits     = model(eeg, ecg)
                    probs      = torch.softmax(logits, dim=1)[:, 1]
                    y_true_v.extend(labels.numpy())
                    y_prob_v.extend(probs.cpu().numpy())

            y_true_v = np.array(y_true_v)
            y_prob_v = np.array(y_prob_v)

            if len(np.unique(y_true_v)) < 2:
                return 0.0

            auc = float(roc_auc_score(y_true_v, y_prob_v))

            # Optuna intermediate reporting + pruning
            trial.report(auc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if auc > best_auc:
                best_auc     = auc
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    break

        return best_auc

    return objective

In [ ]:
N_TRIALS  = 30   # ← increase to 50+ for better results
optuna_all = {}

for target in TARGETS:
    print(f"\n{'#'*55}")
    print(f"  Optuna Search | target={target.upper()} | trials={N_TRIALS}")
    print(f"{'#'*55}")

    study = optuna.create_study(
        direction  = "maximize",
        sampler    = optuna.samplers.TPESampler(seed=42),
        pruner     = optuna.pruners.MedianPruner(
                         n_warmup_steps=5, n_min_trials=10
                     ),
        study_name = f"fusion_{target}",
    )

    study.optimize(
        make_objective(target),
        n_trials         = N_TRIALS,
        show_progress_bar= True,
        gc_after_trial   = True,
    )

    best = study.best_trial
    result = {
        "target"      : target,
        "best_auc"    : round(best.value, 4),
        "best_params" : best.params,
        "n_trials"    : N_TRIALS,
        "n_completed" : len([t for t in study.trials
                             if t.state == optuna.trial.TrialState.COMPLETE]),
        "n_pruned"    : len([t for t in study.trials
                             if t.state == optuna.trial.TrialState.PRUNED]),
    }
    optuna_all[target] = result

    path = f"{RESULTS_DIR}/optuna_fusion_{target}.json"
    with open(path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"\n  ✅ Best AUC   : {result['best_auc']}")
    print(f"     Params     : {result['best_params']}")
    print(f"     Completed  : {result['n_completed']} | "
          f"Pruned: {result['n_pruned']}")

## B1 — Optuna Results Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["#4C72B0", "#DD8452", "#55A868"]

for ax, target, color in zip(axes, TARGETS, colors):
    if target not in optuna_all:
        ax.set_title(f"{target} — no data")
        continue

    path = f"{RESULTS_DIR}/optuna_fusion_{target}.json"
    with open(path) as f:
        res = json.load(f)

    # Reconstruct trial AUC history from completed trials
    # (re-run study in no_storage mode to recover history)
    best_auc = res["best_auc"]
    params   = res["best_params"]

    labels_plot = list(params.keys())
    values_plot = [str(v) for v in params.values()]

    ax.barh(labels_plot, [abs(hash(v)) % 100 / 100 for v in values_plot],
            color=color, alpha=0.7, edgecolor="white")
    ax.set_title(f"{target.capitalize()}\nBest AUC = {best_auc:.4f}",
                 fontweight="bold")
    ax.set_xlabel("Relative param value (scaled)")

    # Annotate actual values
    for i, (lbl, val) in enumerate(zip(labels_plot, values_plot)):
        ax.text(0.02, i, val, va="center", fontsize=8, color="black")

plt.suptitle("Optuna Best Hyperparameters per Target",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/optuna_best_params.png", bbox_inches="tight")
plt.show()

In [ ]:
try:
    import optuna.visualization as ov
    # Only works if plotly is installed
    !pip install plotly -q

    for target in TARGETS:
        study_path = f"{RESULTS_DIR}/optuna_fusion_{target}.json"
        if os.path.exists(study_path):
            print(f"Param importance for {target}: "
                  "re-run study with storage to get full importance plot")
except Exception:
    pass

# ── Fallback: manual param importance from best params table ──────
rows = []
for target in TARGETS:
    if target in optuna_all:
        row = {"target": target,
               "best_auc": optuna_all[target]["best_auc"]}
        row.update(optuna_all[target]["best_params"])
        rows.append(row)

df_optuna = pd.DataFrame(rows)
print("\n── Optuna Best Hyperparameter Table ──")
print(df_optuna.to_string(index=False))
df_optuna.to_csv(f"{RESULTS_DIR}/optuna_best_params_table.csv", index=False)

## B2 — Retrain FusionModel with Best Optuna Params

In [ ]:
from src.data.dataset      import DREAMERDataset
from src.training.trainer  import Trainer

for target in TARGETS:
    if target not in optuna_all:
        print(f"⚠ No Optuna result for {target}, skipping")
        continue

    best_params = optuna_all[target]["best_params"]
    print(f"\n{'='*52}")
    print(f"  Retraining best FusionModel | {target.upper()}")
    print(f"  Params: {best_params}")
    print(f"{'='*52}")

    # Override config with best params
    tuned_config = {
        "data": {
            **config["data"],
            "segment_length": float(best_params.get("window_sec", 4)),
            "overlap"       : float(best_params.get("window_sec", 4)) / 2,
        },
        "model": {
            "type"       : "fusion",
            "branch_dim" : best_params.get("branch_dim", 128),
            "dropout"    : best_params.get("dropout", 0.4),
        },
        "training": {
            **config["training"],
            "learning_rate": best_params.get("lr", 0.001),
            "batch_size"   : best_params.get("batch_size", 32),
            "weight_decay" : best_params.get("weight_decay", 1e-4),
            "epochs"       : best_params.get("n_epochs", 50),
        },
        "labels": config["labels"],
    }

    dataset = DREAMERDataset(
        mat_path    = config["data"]["raw_path"],
        target      = target,
        window_sec  = tuned_config["data"]["segment_length"],
        overlap_sec = tuned_config["data"]["overlap"],
        norm_method = config["data"]["norm_method"],
        threshold   = float(config["labels"]["threshold"]),
    )
    dataset.summary()

    from src.models.deep_model import build_model
    model = build_model("fusion", tuned_config)

    trainer = Trainer(
        model,
        dataset,
        tuned_config,
        target        = target,
        checkpoint_dir= MODELS_DIR,
    )
    history = trainer.fit()
    trainer.load_best()

    print(f"\n  ✅ Best tuned model saved for {target}")

## C — Tuning Summary: Baseline vs Deep Best AUC

In [ ]:
summary_rows = []

# Baseline best (from grid search)
for target in TARGETS:
    best_bl_auc  = 0.0
    best_bl_mdl  = ""
    for mtype in ["rf", "svm", "logreg", "gbm"]:
        p = f"{RESULTS_DIR}/gridsearch_{mtype}_{target}.json"
        if os.path.exists(p):
            with open(p) as f:
                d = json.load(f)
            if d["best_auc"] > best_bl_auc:
                best_bl_auc = d["best_auc"]
                best_bl_mdl = mtype.upper()

    summary_rows.append({
        "target"    : target.capitalize(),
        "type"      : f"Baseline ({best_bl_mdl})",
        "best_auc"  : best_bl_auc,
        "source"    : "GridSearchCV (5-fold CV)",
    })

# Deep best (from Optuna)
for target in TARGETS:
    if target in optuna_all:
        summary_rows.append({
            "target"    : target.capitalize(),
            "type"      : "Deep (FusionModel)",
            "best_auc"  : optuna_all[target]["best_auc"],
            "source"    : f"Optuna TPE ({N_TRIALS} trials)",
        })

df_final = pd.DataFrame(summary_rows)
print("── Final Tuning Summary ──")
print(df_final.to_string(index=False))
df_final.to_csv(f"{RESULTS_DIR}/tuning_final_summary.csv", index=False)

# ── Bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x       = np.arange(len(TARGETS))
w       = 0.35

bl_aucs = df_final[df_final["type"].str.startswith("Baseline")]["best_auc"].values
dp_aucs = df_final[df_final["type"].str.startswith("Deep")]["best_auc"].values

ax.bar(x - w/2, bl_aucs, w, label="Baseline (GridSearch)",
       color="#4C72B0", edgecolor="white")
ax.bar(x + w/2, dp_aucs, w, label="Deep (Optuna)",
       color="#DD8452", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels([t.capitalize() for t in TARGETS], fontweight="bold")
ax.set_ylabel("Best ROC-AUC")
ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color="black", linestyle="--",
           linewidth=0.8, alpha=0.5, label="Chance")
ax.legend()
ax.set_title("Tuned Performance: Baseline vs Deep",
             fontweight="bold", fontsize=13)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/tuning_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
if IN_COLAB:
    !git config --global user.email "you@example.com"
    !git config --global user.name "Your Name"
    !git add outputs/results/ outputs/models/ notebooks/
    !git commit -m "feat: hyperparameter tuning — GridSearch + Optuna results"
    !git push
    print("✅ Pushed to GitHub")